In [1]:
!pip install -q decord

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 52.0 MB/s eta 0:00:00


In [2]:
import os, math, random, tarfile
from pathlib import Path
import numpy as np
import imageio.v2 as imageio
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from IPython.display import Video,display

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset,DataLoader
from decord import VideoReader,cpu
from huggingface_hub import hf_hub_download
from diffusers import DDPMScheduler, DDIMScheduler

In [3]:
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed(SEED)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
# assert device == 'cuda', 'Select Runtime > Change runtime type > GPU in Colab.'


In [4]:
NUM_FRAMES = 16
IMAGE_SIZE = 64
PATCH_T, PATCH_H, PATCH_W = 2, 8, 8
MODEL_DIM, NUM_HEADS, NUM_LAYERS = 384, 6, 6
BATCH_SIZE, GRAD_ACCUM = 2, 8
NUM_DIFFUSION_STEPS = 1000
CFG_DROPOUT = 0.15
TRAIN_STEPS = 5000                 # use 15000-30000 for better results
LR = 2e-4
DATA_ROOT = Path('/content/ucf101_data')
PREVIEW_ROOT = Path('/content/video_dit_previews')
DATA_ROOT.mkdir(exist_ok=True); PREVIEW_ROOT.mkdir(exist_ok=True)

In [5]:
archive = hf_hub_download(
    repo_id='sayakpaul/ucf101-subset',
    filename='UCF101_subset.tar.gz',
    repo_type='dataset'
)

if not list(DATA_ROOT.rglob('*.avi')) and not list(DATA_ROOT.rglob('*.mp4')):
    with tarfile.open(archive) as tar:
        tar.extractall(path=DATA_ROOT)

roots = [p for p in DATA_ROOT.rglob('*') if p.is_dir() and (p/'train').exists()]
DATASET_ROOT = roots[0] if roots else DATA_ROOT

UCF101_subset.tar.gz: reconstructing file:   0%|          |  0.00B /  171MB            

UCF101_subset.tar.gz: downloading bytes:           |  0.00B            

/tmp/ipykernel_2383/549539243.py:9: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=DATA_ROOT)


In [6]:
DATASET_ROOT

PosixPath('/content/ucf101_data/UCF101_subset')

In [13]:
def read_clip(path, train=True):
    vr = VideoReader(str(path), ctx=cpu(0))
    n = len(vr)
    if n < NUM_FRAMES:
        ids = np.linspace(0, max(n-1, 0), NUM_FRAMES).astype(np.int64)
    elif train:
        start = random.randint(0, n-NUM_FRAMES)
        ids = np.arange(start, start+NUM_FRAMES)
    else:
        ids = np.linspace(0, n-1, NUM_FRAMES).astype(np.int64)
    frames = torch.from_numpy(vr.get_batch(ids).asnumpy()).float()/255.
    x = frames.permute(0, 3, 1, 2)
    _, _, h, w = x.shape
    scale = IMAGE_SIZE/min(h, w)
    nh, nw = round(h*scale), round(w*scale)
    x = F.interpolate(x, size=(nh, nw), mode='bilinear', align_corners=False)
    top, left = max((nh-IMAGE_SIZE)//2, 0), max((nw-IMAGE_SIZE)//2, 0)
    x = x[:, :, top:top+IMAGE_SIZE, left:left+IMAGE_SIZE]
    if train and random.random() < .5: x = torch.flip(x, dims=[3])
    return (x*2-1).permute(1, 0, 2, 3).contiguous()

In [14]:
class UCFVideoDataset(Dataset):
    def __init__(self, root, split, train):
        self.train = train
        base = Path(root)/split
        self.files = sorted([p for ext in ('*.avi','*.mp4','*.mov','*.webm') for p in base.rglob(ext)])
        self.classes = sorted({p.parent.name for p in self.files})
        self.class_to_idx = {c:i for i,c in enumerate(self.classes)}
        self.labels = [self.class_to_idx[p.parent.name] for p in self.files]
        print(split, len(self.files), 'videos,', len(self.classes), 'classes')
    def __len__(self): return len(self.files)
    def __getitem__(self, i):
        try: x = read_clip(self.files[i], self.train)
        except Exception: return self.__getitem__(random.randrange(len(self.files)))
        return {'video':x, 'label':torch.tensor(self.labels[i]), 'path':str(self.files[i])}

In [15]:
train_ds = UCFVideoDataset(DATASET_ROOT,'train',True)
val_split = 'val' if (DATASET_ROOT/'val').exists() else 'test'
val_ds = UCFVideoDataset(DATASET_ROOT,val_split,False)
CLASS_NAMES,NUM_CLASSES = train_ds.classes,len(train_ds.classes)
train_loader = DataLoader(train_ds,BATCH_SIZE,shuffle=True,num_workers=2,pin_memory=True,drop_last=True)
val_loader = DataLoader(val_ds,BATCH_SIZE,shuffle=False,num_workers=2,pin_memory=True)

train 300 videos, 10 classes
val 30 videos, 10 classes


In [16]:
def save_video(x, path, fps=8):
    frames = ((x.detach().cpu()+1)*127.5).clamp(0,255).byte().permute(1,2,3,0).numpy()
    writer = imageio.get_writer(path, fps=fps, codec='libx264', quality=7)
    for frame in frames: writer.append_data(frame)
    writer.close(); return path

batch = next(iter(val_loader))
for i in range(min(2, len(batch['video']))):
    p = PREVIEW_ROOT/f'test_{i}.mp4'
    save_video(batch['video'][i], p)
    print(CLASS_NAMES[batch['label'][i].item()]); display(Video(str(p), embed=True))


ApplyEyeMakeup


ApplyEyeMakeup


In [17]:
class TimeEmbedding(nn.Module):
    def __init__(self, dim): super().__init__(); self.dim = dim
    def forward(self, t):
        half = self.dim//2
        freq = torch.exp(-math.log(10000)*torch.arange(half, device=t.device)/(max(half-1,1)))
        a = t.float()[:,None]*freq[None]
        return torch.cat([a.sin(), a.cos()], -1)

In [18]:
class Attention(nn.Module):
    def __init__(self, dim, heads):
        super().__init__(); self.heads=heads; self.d=dim//heads
        self.qkv=nn.Linear(dim, 3*dim); self.proj=nn.Linear(dim, dim)
    def forward(self,x):
        b,n,c=x.shape; qkv=self.qkv(x).reshape(b,n,3,self.heads,self.d).permute(2,0,3,1,4)
        q,k,v=qkv.unbind(0)
        y=F.scaled_dot_product_attention(q,k,v,dropout_p=0.0)
        return self.proj(y.transpose(1,2).reshape(b,n,c))

In [19]:
class Block(nn.Module):
    def __init__(self, dim, heads):
        super().__init__(); self.n1=nn.LayerNorm(dim, elementwise_affine=False); self.n2=nn.LayerNorm(dim, elementwise_affine=False)
        self.attn=Attention(dim,heads); hidden=4*dim
        self.mlp=nn.Sequential(nn.Linear(dim,hidden),nn.GELU(approximate='tanh'),nn.Linear(hidden,dim))
        self.ada=nn.Sequential(nn.SiLU(),nn.Linear(dim,6*dim))
        nn.init.zeros_(self.ada[-1].weight); nn.init.zeros_(self.ada[-1].bias)
    def forward(self,x,c):
        s1,a1,g1,s2,a2,g2=self.ada(c).chunk(6,-1)
        x=x+g1[:,None]*self.attn(self.n1(x)*(1+a1[:,None])+s1[:,None])
        x=x+g2[:,None]*self.mlp(self.n2(x)*(1+a2[:,None])+s2[:,None])
        return x

In [20]:
class VideoDiT(nn.Module):
    def __init__(self, num_classes, dim=MODEL_DIM, depth=NUM_LAYERS, heads=NUM_HEADS):
        super().__init__(); self.dim=dim
        self.patch_embed=nn.Conv3d(3,dim,(PATCH_T,PATCH_H,PATCH_W),(PATCH_T,PATCH_H,PATCH_W))
        nt=(NUM_FRAMES//PATCH_T)*(IMAGE_SIZE//PATCH_H)*(IMAGE_SIZE//PATCH_W)
        self.pos=nn.Parameter(torch.randn(1,nt,dim)*.02)
        self.t=nn.Sequential(TimeEmbedding(dim),nn.Linear(dim,4*dim),nn.SiLU(),nn.Linear(4*dim,dim))
        self.cls=nn.Embedding(num_classes+1,dim)
        self.blocks=nn.ModuleList([Block(dim,heads) for _ in range(depth)])
        self.norm=nn.LayerNorm(dim); self.head=nn.Linear(dim,3*PATCH_T*PATCH_H*PATCH_W)
        nn.init.zeros_(self.head.weight); nn.init.zeros_(self.head.bias)
    def forward(self,x,t,labels):
        b=x.shape[0]; h=self.patch_embed(x).flatten(2).transpose(1,2)+self.pos
        c=self.t(t)+self.cls(labels)
        for block in self.blocks: h=block(h,c)
        h=self.head(self.norm(h)).reshape(b,NUM_FRAMES//PATCH_T,IMAGE_SIZE//PATCH_H,IMAGE_SIZE//PATCH_W,3,PATCH_T,PATCH_H,PATCH_W)
        return h.permute(0,4,1,5,2,6,3,7).reshape(b,3,NUM_FRAMES,IMAGE_SIZE,IMAGE_SIZE)

In [21]:
model=VideoDiT(NUM_CLASSES).to(device)
print('parameters:', sum(p.numel() for p in model.parameters())/1e6, 'M')
NULL_CLASS=NUM_CLASSES
train_scheduler=DDPMScheduler(num_train_timesteps=NUM_DIFFUSION_STEPS,beta_schedule='squaredcos_cap_v2',prediction_type='epsilon',clip_sample=False)
sample_scheduler=DDIMScheduler(num_train_timesteps=NUM_DIFFUSION_STEPS,beta_schedule='squaredcos_cap_v2',prediction_type='epsilon',clip_sample=False,set_alpha_to_one=False)
optimizer=torch.optim.AdamW(model.parameters(),lr=LR,betas=(.9,.95),weight_decay=.01)
scaler=torch.amp.GradScaler('cuda')

parameters: 17.638656 M


In [22]:
train_iter=iter(train_loader); losses=[]; optimizer.zero_grad(set_to_none=True)
for step in tqdm(range(1, TRAIN_STEPS+1)):
    try: batch=next(train_iter)
    except StopIteration: train_iter=iter(train_loader); batch=next(train_iter)
    x0=batch['video'].to(device, non_blocking=True); labels=batch['label'].to(device, non_blocking=True)
    t=torch.randint(0,NUM_DIFFUSION_STEPS,(x0.shape[0],),device=device)
    noise=torch.randn_like(x0); xt=train_scheduler.add_noise(x0,noise,t)
    labels_in=labels.clone(); labels_in[torch.rand(x0.shape[0],device=device)<CFG_DROPOUT]=NULL_CLASS
    with torch.autocast('cuda',dtype=torch.float16):
        pred=model(xt,t,labels_in); loss=F.mse_loss(pred.float(),noise.float())/GRAD_ACCUM
    scaler.scale(loss).backward()
    if step%GRAD_ACCUM==0:
        scaler.unscale_(optimizer); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
        scaler.step(optimizer); scaler.update(); optimizer.zero_grad(set_to_none=True)
    losses.append(loss.item()*GRAD_ACCUM)
    if step%100==0: print('step',step,'loss',float(np.mean(losses[-100:])))
    if step%2000==0: torch.save({'model':model.state_dict(),'optimizer':optimizer.state_dict(),'classes':CLASS_NAMES,'step':step},f'/content/video_dit_step_{step}.pt')


  0%|          | 0/5000 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7953290a5120>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7953290a5120>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 16

step 100 loss 0.9697858792543411
step 200 loss 0.9005399018526077
step 300 loss 0.8319993537664413
step 400 loss 0.7754646843671799
step 500 loss 0.7137151098251343
step 600 loss 0.6675060427188874
step 700 loss 0.6160911577939987
step 800 loss 0.5709228342771531
step 900 loss 0.542070409655571
step 1000 loss 0.5074087259173393
step 1100 loss 0.47095291525125504
step 1200 loss 0.42813079208135607
step 1300 loss 0.40983176440000535
step 1400 loss 0.36448545604944227
step 1500 loss 0.3297255700826645
step 1600 loss 0.31721611648797987
step 1700 loss 0.31001055628061297
step 1800 loss 0.2949133698642254
step 1900 loss 0.2661112579703331
step 2000 loss 0.2932939536869526
step 2100 loss 0.26949624955654145
step 2200 loss 0.26913660123944283
step 2300 loss 0.2354108838737011
step 2400 loss 0.20440857991576195
step 2500 loss 0.19698603466153145
step 2600 loss 0.18781798012554646
step 2700 loss 0.2091931787133217
step 2800 loss 0.2179571471363306
step 2900 loss 0.18227315455675125
step 3000 lo

In [24]:
@torch.no_grad()
def sample_video(labels, steps=50, guidance=3.5):
    model.eval(); b=labels.shape[0]
    x=torch.randn(b,3,NUM_FRAMES,IMAGE_SIZE,IMAGE_SIZE,device=device)
    sample_scheduler.set_timesteps(steps,device=device)
    for ts in tqdm(sample_scheduler.timesteps,desc='sampling'):
        t=torch.full((b,),int(ts.item()),device=device,dtype=torch.long)
        null=torch.full_like(labels,NULL_CLASS)
        with torch.autocast('cuda',dtype=torch.float16):
            cond=model(x,t,labels); uncond=model(x,t,null)
        eps=uncond+guidance*(cond-uncond)
        x=sample_scheduler.step(eps,ts,x).prev_sample
    return x.clamp(-1,1)

In [25]:
labels=torch.arange(NUM_CLASSES,device=device)
generated=sample_video(labels,steps=50,guidance=3.5)
for i,x in enumerate(generated):
    p=PREVIEW_ROOT/f'generated_{i}_{CLASS_NAMES[i]}.mp4'
    save_video(x,p); print(CLASS_NAMES[i]); display(Video(str(p),embed=True))

# checkpoint
torch.save({'model':model.state_dict(),'optimizer':optimizer.state_dict(),'classes':CLASS_NAMES,'config':{'frames':NUM_FRAMES,'size':IMAGE_SIZE,'dim':MODEL_DIM,'layers':NUM_LAYERS,'heads':NUM_HEADS}},'/content/video_dit_final.pt')

sampling:   0%|          | 0/50 [00:00<?, ?it/s]

ApplyEyeMakeup


ApplyLipstick


Archery


BabyCrawling


...

BandMarching


BaseballPitch


Basketball


BasketballDunk


BenchPress
